In [0]:
# Create streaming source from jobs Bronze
from pyspark.sql import functions as F

jobs = spark.table("ecommerce.bronze.jobs_bronze")

# Write 3 batches to a Delta table to simulate live stream
for i in range(3):
    jobs.limit(100) \
        .withColumn("batch_id", F.lit(i)) \
        .withColumn("ingest_time", F.current_timestamp()) \
        .write.format("delta") \
        .mode("append") \
        .saveAsTable("ecommerce.bronze.stream_source")
    print(f"Batch {i+1} written ✓")

Batch 1 written ✓
Batch 2 written ✓
Batch 3 written ✓


In [0]:
# Read stream from Delta table
live_stream = spark.readStream \
    .format("delta") \
    .table("ecommerce.bronze.stream_source")

print("Stream reader created ✓")

Stream reader created ✓


In [0]:
# Aggregate live job demand
live_demand = live_stream.groupBy(
    "job_category",
    "experience_level",
    "work_setting"
).agg(
    F.count("*").alias("live_job_count"),
    F.avg("salary_in_usd").alias("live_avg_salary")
)

In [0]:
# Write stream to Gold Delta table
import time

query = live_demand.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", "/Volumes/ecommerce/gold/ml_models/checkpoint") \
    .trigger(availableNow=True) \
    .toTable("ecommerce.gold.live_demand_gold")

print("Streaming running ✓")
time.sleep(30)
query.stop()
print("Streaming complete ✓")

Streaming running ✓
Streaming complete ✓


In [0]:
# Show Gold results
print("Live demand Gold table:")
spark.table("ecommerce.gold.live_demand_gold") \
    .orderBy(F.desc("live_job_count")) \
    .show(10, truncate=False)

Live demand Gold table:
+------------------------------+----------------+------------+--------------+------------------+
|job_category                  |experience_level|work_setting|live_job_count|live_avg_salary   |
+------------------------------+----------------+------------+--------------+------------------+
|Data Science and Research     |Senior          |In-person   |102           |190679.70588235295|
|Data Engineering              |Senior          |In-person   |96            |146714.4375       |
|Machine Learning and AI       |Senior          |In-person   |60            |199480.0          |
|Machine Learning and AI       |Mid-level       |In-person   |48            |182525.0          |
|Data Engineering              |Mid-level       |In-person   |36            |128900.0          |
|Data Science and Research     |Senior          |Remote      |36            |168266.66666666666|
|Data Science and Research     |Mid-level       |In-person   |36            |114696.0          |
|Data 

In [0]:
# Final project summary
print("=" * 55)
print("  INDIA YOUTH EMPLOYMENT INTELLIGENCE PLATFORM")
print("  Final Summary")
print("=" * 55)

flagged = spark.table(
    "ecommerce.gold.state_predictions_gold"
).filter(F.col("prediction") == 1).count()

top_state = spark.table(
    "ecommerce.gold.state_predictions_gold"
).orderBy(F.desc("avg_unemployment_rate")).first()

top_cat = spark.table(
    "ecommerce.gold.live_demand_gold"
).orderBy(F.desc("live_job_count")).first()

print(f"\nBronze layer  : unemployment + jobs Delta tables ✓")
print(f"Silver layer  : 5 feature tables ✓")
print(f"ML model      : GBT Classifier AUC = 1.0 ✓")
print(f"Gold layer    : predictions + live demand ✓")
print(f"\nStates needing intervention : {flagged}")
print(f"Highest unemployment state  : {top_state['region']} "
      f"({top_state['avg_unemployment_rate']:.1f}%)")
print(f"Most in-demand job category : {top_cat['job_category']}")
print("\nProject complete ✓")

  INDIA YOUTH EMPLOYMENT INTELLIGENCE PLATFORM
  Final Summary

Bronze layer  : unemployment + jobs Delta tables ✓
Silver layer  : 5 feature tables ✓
ML model      : GBT Classifier AUC = 1.0 ✓
Gold layer    : predictions + live demand ✓

States needing intervention : 2
Highest unemployment state  : Haryana (27.5%)
Most in-demand job category : Data Science and Research

Project complete ✓
